<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 9 · Comparado con qué: variabilidad, comparación y causalidad

La semana pasada dejaste a los clientes repartidos en seis segmentos con nombre y mensaje comercial.
Hoy llega la pregunta que ese trabajo no contesta: **cuando un número sube o baja, ¿pasó algo o es la
variación de siempre?** Y la que viene detrás, que es peor: cuando dos grupos difieren, ¿es por lo que
estás midiendo o por algo que ni miraste? Empiezas con el experimento de reactivación de Comercial
Andina, donde la campaña **parece hacer daño**. Al abrirlo por tipo de cliente el resultado se da la
vuelta entero. Ese giro tiene nombre —paradoja de Simpson— y es la razón por la que una tabla correcta
puede sostener la decisión contraria a la correcta.

> **Hoy haces** · Encuentras y explicas la paradoja de Simpson del experimento de reactivación, con la
> tabla cruzada que muestra por qué se invierte (90 min). Aplicas las tres pruebas de uso diario sobre
> Comercial Andina —t para dos grupos, chi cuadrado para dos categóricas, ANOVA para cinco ciudades—,
> calculas el tamaño del efecto de cada una y traduces los resultados a una frase de negocio. Cierras
> diseñando la prueba A/B de la campaña con el tamaño de muestra calculado y el criterio de parada
> escrito antes de lanzarla.
>
> **Entrega** · Este cuaderno ejecutado, la explicación escrita de la paradoja con la conclusión de
> negocio, la tabla de las pruebas con p-valor **y** tamaño del efecto, y la plantilla de diseño A/B
> del negocio del caso completa con sus seis campos.
> Nombre de archivo: `lab_09_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              Path("/content/CursoAnalisisDatos_IA_2026/sitio/datos")]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    raise FileNotFoundError(
        "No encuentro la carpeta de datos. En Colab ejecuta primero:\n"
        "  !git clone https://github.com/<usuario>/CursoAnalisisDatos_IA_2026.git")

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. El experimento que dice que la campaña hace daño

Comercial Andina mandó una campaña de reactivación por correo a parte de su cartera y dejó al resto
sin tocar. El archivo `experimento_reactivacion.csv` tiene una fila por cliente: en qué grupo cayó y
si volvió a comprar dentro de la ventana. Es lo más parecido a un experimento que tiene la empresa, y
la primera lectura es demoledora.

In [ ]:
experimento = pd.read_csv(DATOS / "experimento_reactivacion.csv")

global_ = (experimento.groupby("grupo")["convirtio"]
           .agg(clientes="size", conversiones="sum", tasa="mean"))
global_["tasa %"] = global_["tasa"] * 100

print(f"{len(experimento):,} clientes en el experimento · "
      f"{experimento['convirtio'].sum():,} conversiones\n")
print(global_.round(4).to_string())

dif = global_.loc["Tratamiento", "tasa"] - global_.loc["Control", "tasa"]
print(f"\nefecto aparente de la campaña: {dif * 100:+.2f} puntos porcentuales")
print(f"lectura ingenua: la campaña convierte "
      f"{global_.loc['Tratamiento', 'tasa'] / global_.loc['Control', 'tasa'] - 1:+.1%} "
      f"que no hacer nada")

Con esa tabla encima de la mesa, la recomendación se escribe sola: **apagar la campaña**. El grupo
que no recibió nada convierte 21,76 % y el que recibió el correo, 14,15 %. Siete puntos y medio de
diferencia, sobre 1 800 clientes. No es un margen ambiguo.

Antes de firmarla, una sola pregunta: **¿los dos grupos eran comparables?** Es la pregunta de toda la
semana y la que casi nadie hace, porque la palabra «experimento» ya suena a que alguien se encargó.

## 2. La paradoja de Simpson

Abrir el resultado por un tercer factor no es buscar hasta que salga lo que quieres: es comprobar que
la comparación era legítima. Aquí el tercer factor es obvio porque el curso lleva ocho semanas
diciéndolo: en Comercial Andina hay dos negocios metidos en la misma tabla, el minorista y el
mayorista.

In [ ]:
por_segmento = (experimento.groupby(["tipo_cliente", "grupo"])["convirtio"]
                .agg(clientes="size", conversiones="sum", tasa="mean"))
por_segmento["tasa %"] = por_segmento["tasa"] * 100
print(por_segmento.round(4).to_string(), "\n")

tasas = experimento.pivot_table(index="tipo_cliente", columns="grupo",
                                values="convirtio", aggfunc="mean")
tasas["diferencia pp"] = (tasas["Tratamiento"] - tasas["Control"]) * 100

fila_global = pd.DataFrame({
    "Control": [experimento.query("grupo == 'Control'")["convirtio"].mean()],
    "Tratamiento": [experimento.query("grupo == 'Tratamiento'")["convirtio"].mean()],
}, index=["TODOS JUNTOS"])
fila_global["diferencia pp"] = (fila_global["Tratamiento"] - fila_global["Control"]) * 100

print((pd.concat([tasas, fila_global]) * [100, 100, 1]).round(2).to_string())
print("\nGana el tratamiento en cada segmento por separado y pierde en el total.")

In [ ]:
# ✅ Comprobación 1 · la paradoja tiene que aparecer en los dos sentidos a la vez
tasas_g = experimento.groupby("grupo")["convirtio"].mean()
por_tipo = experimento.pivot_table(index="tipo_cliente", columns="grupo",
                                   values="convirtio", aggfunc="mean")

assert tasas_g["Control"] > tasas_g["Tratamiento"], \
    "En el total, el control tiene que ganar: si no, revisa que leíste experimento_reactivacion.csv"
assert (por_tipo["Tratamiento"] > por_tipo["Control"]).all(), \
    "Dentro de cada tipo de cliente tiene que ganar el tratamiento: ahí está la paradoja"
assert abs(tasas_g["Control"] - 0.218) < 0.01 and abs(tasas_g["Tratamiento"] - 0.141) < 0.01, \
    "Las tasas globales del curso son 21,8 % de control y 14,1 % de tratamiento: no coinciden"

print("Comprobación 1 superada ✓  el control gana en total y pierde en los dos segmentos: es Simpson")

📌 **La campaña gana en los dos segmentos y pierde en el total.** En minoristas sube la conversión
del 5,62 % al 11,17 % —5,55 puntos, casi el doble— y en mayoristas del 39,35 % al 55,13 %, otros 15,77
puntos. Sumados los dos, pierde por 7,61 puntos.

No es un error de cálculo ni un truco: las cinco cifras son correctas. Es la **paradoja de Simpson**, y
ocurre cuando el tercer factor cumple dos condiciones a la vez. Aquí las cumple con creces.

In [ ]:
# Condición 1 · el tercer factor cambia mucho el resultado, independientemente del grupo.
base = experimento.groupby("tipo_cliente")["convirtio"].mean()
print("Condición 1 · ¿el tipo de cliente cambia la conversión por sí solo?")
print(f"  Minorista {base['Minorista']:.2%}  ·  Mayorista {base['Mayorista']:.2%}  "
      f"→ el mayorista convierte {base['Mayorista'] / base['Minorista']:.1f} veces más\n")

# Condición 2 · el tercer factor está repartido de forma desigual entre los grupos.
asignacion = pd.crosstab(experimento["tipo_cliente"], experimento["grupo"], margins=True,
                         margins_name="Total")
reparto = pd.crosstab(experimento["tipo_cliente"], experimento["grupo"], normalize="index") * 100

print("Condición 2 · ¿cómo quedó repartido cada segmento entre los dos grupos?")
print(asignacion.to_string(), "\n")
print(reparto.round(2).to_string())
print("\nEl 79,90 % de los mayoristas cayó en control y el 76,06 % de los minoristas en tratamiento.")
print("Con una asignación al azar los dos porcentajes deberían rondar el 50 %.")

Ahí está todo el mecanismo. Los mayoristas convierten **4,3 veces más** que los minoristas pase lo
que pase —42,53 % contra 9,84 %— y ocho de cada diez mayoristas quedaron en el grupo de control. El control no ganó
porque no recibir el correo funcione: ganó porque le tocaron los clientes fáciles.

La aleatorización es exactamente la herramienta que evita esto, y aquí no funcionó. Se comprueba con
una prueba, no con el ojo.

In [ ]:
from scipy import stats

chi2_asig, p_asig, gl_asig, esperados = stats.chi2_contingency(
    pd.crosstab(experimento["tipo_cliente"], experimento["grupo"]))

print("¿La asignación a los grupos fue independiente del tipo de cliente?")
print(f"  chi cuadrado = {chi2_asig:,.4f}   grados de libertad = {gl_asig}   p = {p_asig:.3g}")
print("\nMayoristas y minoristas por grupo, observados frente a los esperados si fuera al azar:")
comparacion = pd.DataFrame(
    np.hstack([pd.crosstab(experimento["tipo_cliente"], experimento["grupo"]).values,
               esperados.round(1)]),
    index=["Mayorista", "Minorista"],
    columns=["Control obs.", "Tratamiento obs.", "Control esp.", "Tratamiento esp."])
print(comparacion.to_string())
print("\nUn p de 1,96e-91 no es «hay indicios de desbalance»: es que este reparto no sale al azar.")

⚠️ **El experimento estaba roto antes de empezar.** Alguien armó las listas a mano, o el sistema
excluyó del envío a los clientes con crédito abierto —que son los mayoristas—, o se mandó primero a
quien tenía correo registrado. Da igual cuál de las tres: el resultado global no mide la campaña, mide
la mezcla.

Lo que sí se puede rescatar es el efecto dentro de cada segmento, donde la comparación sí es limpia. Y
se puede reconstruir el número global correcto **estandarizando**: calcular qué habría pasado si los
dos grupos hubieran tenido la misma composición que la cartera real.

In [ ]:
# Estandarización directa: se aplican las tasas de cada grupo a la composición REAL de la cartera.
mezcla_real = experimento["tipo_cliente"].value_counts(normalize=True)
estandarizado = (tasas[["Control", "Tratamiento"]] * mezcla_real.reindex(tasas.index).values[:, None]).sum()

print("Composición real de la cartera:")
print(f"  {mezcla_real.to_dict()}\n")
print(f"tasa ingenua      · control {global_.loc['Control', 'tasa']:.4%} vs "
      f"tratamiento {global_.loc['Tratamiento', 'tasa']:.4%}  "
      f"→ {dif * 100:+.2f} pp")
print(f"tasa estandarizada· control {estandarizado['Control']:.4%} vs "
      f"tratamiento {estandarizado['Tratamiento']:.4%}  "
      f"→ {(estandarizado['Tratamiento'] - estandarizado['Control']) * 100:+.2f} pp")
print(f"\nEl mismo experimento, los mismos 1 800 clientes: "
      f"{dif * 100:+.2f} pp o {(estandarizado['Tratamiento'] - estandarizado['Control']) * 100:+.2f} pp "
      f"según se controle o no la mezcla.")

### La conclusión de negocio, en tres frases

1. **La campaña funciona y se mantiene.** Sube la conversión 5,55 puntos en minoristas y 15,77 en
   mayoristas. Corregida por la mezcla real de la cartera, el efecto global es de **+7,76 puntos**, no
   de −7,61. La recomendación de apagarla habría destruido lo único que funcionaba.
2. **El experimento no se puede repetir así.** La asignación quedó atada al tipo de cliente
   (p = 1,96e-91). El próximo se aleatoriza **estratificando por `tipo_cliente`**.
3. **El presupuesto va donde el efecto es grande.** Los 15,77 puntos de los mayoristas se consiguen
   sobre una base que ya convierte al 39 %; los 5,55 de los minoristas casi duplican su conversión. Con
   presupuesto limitado, el orden lo decide cuánto vale cada conversión, no cuál porcentaje suena mejor.

Y la regla general: **antes de comparar dos grupos, comprueba que se parecen en todo lo demás.** Una
tabla de dos filas nunca alcanza para eso.

## 3. Prueba t: comparar el promedio de dos grupos

La prueba t contesta una sola pregunta: si los dos grupos vinieran del mismo sitio, ¿qué tan raro sería
ver una diferencia como esta? El p-valor es esa rareza. **No es la probabilidad de que la hipótesis sea
falsa** y no dice si la diferencia importa: solo dice si es compatible con el azar.

El caso: Cuenca factura un ticket medio más alto que Manta. ¿Es real?

In [ ]:
ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
clientes = pd.read_csv(DATOS / "clientes.csv")
sucursales = pd.read_csv(DATOS / "sucursales.csv")
clientes["ciudad"] = (clientes["ciudad"].str.strip().str.title()
                      .replace({"Guayaquíl": "Guayaquil"}))
ventas["monto"] = ventas["cantidad"] * ventas["precio_unitario"] * (1 - ventas["descuento"])

# Una fila por factura: la unidad de análisis de hoy es la visita, no la línea.
facturas = (ventas[~ventas["es_devolucion"]]
            .groupby(["factura_id", "cliente_id", "sucursal_id"])["monto"].sum().reset_index()
            .merge(clientes[["cliente_id", "ciudad", "tipo_cliente"]], on="cliente_id",
                   how="left", validate="m:1")
            .merge(sucursales[["sucursal_id", "canal"]], on="sucursal_id",
                   how="left", validate="m:1"))

print(f"{len(facturas):,} facturas · ticket medio global {facturas['monto'].mean():,.2f}\n")
print(facturas.groupby("ciudad")["monto"].agg(["size", "mean", "median", "std"]).round(2).to_string())

def d_de_cohen(a, b):
    """Diferencia de medias expresada en desviaciones estándar."""
    na, nb = len(a), len(b)
    s = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / s


cuenca = facturas.query("ciudad == 'Cuenca'")["monto"]
manta = facturas.query("ciudad == 'Manta'")["monto"]

t, p = stats.ttest_ind(cuenca, manta, equal_var=False)   # Welch: no supone varianzas iguales
d = d_de_cohen(cuenca, manta)

print(f"Cuenca : n = {len(cuenca):,}   media {cuenca.mean():,.2f}   mediana {cuenca.median():,.2f}")
print(f"Manta  : n = {len(manta):,}   media {manta.mean():,.2f}   mediana {manta.median():,.2f}")
print(f"\ndiferencia de medias : {cuenca.mean() - manta.mean():,.2f} dólares por factura")
print(f"estadístico t        : {t:,.4f}")
print(f"p-valor              : {p:.3g}   ← significativo con cualquier umbral")
print(f"d de Cohen           : {d:.4f}   ← efecto pequeño (0,2 pequeño · 0,5 medio · 0,8 grande)")

Significativa y minúscula al mismo tiempo. El p-valor es 1,43e-09 —una entre setecientos
millones— y la d de Cohen es 0,1795, por debajo del umbral de «pequeño». Las dos cosas son ciertas: la
diferencia de 47,71 dólares **existe**, y es diminuta comparada con lo que varía el ticket dentro de
cada ciudad, donde la desviación estándar pasa de 250.

Antes de llevarla a un informe, la misma pregunta de la sección 2: ¿son comparables Cuenca y Manta?

In [ ]:
mezcla = pd.crosstab(facturas["ciudad"], facturas["tipo_cliente"], normalize="index") * 100
print("Qué porcentaje de las facturas de cada ciudad es mayorista:")
print(mezcla.round(2).to_string(), "\n")

# La misma prueba, ahora dentro de cada tipo de cliente.
for tipo in ["Minorista", "Mayorista"]:
    a = facturas.query("ciudad == 'Cuenca' and tipo_cliente == @tipo")["monto"]
    b = facturas.query("ciudad == 'Manta'  and tipo_cliente == @tipo")["monto"]
    t_i, p_i = stats.ttest_ind(a, b, equal_var=False)
    print(f"{tipo:10s} Cuenca {a.mean():>8,.2f} vs Manta {b.mean():>8,.2f}  "
          f"dif {a.mean() - b.mean():>8,.2f}  p = {p_i:.4f}  d = {d_de_cohen(a, b):+.4f}")

# Y el ticket que tendría cada ciudad si todas tuvieran la misma mezcla que el total.
mezcla_pais = facturas["tipo_cliente"].value_counts(normalize=True)
medias = facturas.groupby(["ciudad", "tipo_cliente"])["monto"].mean().unstack()
estandar = (medias * mezcla_pais).sum(axis=1)
tabla = pd.DataFrame({"ticket observado": facturas.groupby("ciudad")["monto"].mean(),
                      "ticket estandarizado": estandar})
tabla["diferencia"] = tabla["ticket observado"] - tabla["ticket estandarizado"]
print("\n", tabla.round(2).to_string(), sep="")
print(f"\nrecorrido entre ciudades · observado {tabla['ticket observado'].max() - tabla['ticket observado'].min():,.2f}"
      f"  ·  estandarizado {tabla['ticket estandarizado'].max() - tabla['ticket estandarizado'].min():,.2f}")

📌 **La diferencia entre Cuenca y Manta era la mezcla, no la ciudad.** El 42,13 % de las facturas de
Cuenca son mayoristas contra el 29,77 % de Manta, y un mayorista deja veinte veces más por visita. Al
comparar dentro de cada tipo de cliente la ventaja de Cuenca desaparece y hasta cambia de signo: entre
mayoristas queda 24,06 dólares **por debajo** de Manta (p = 0,0622) y entre minoristas la diferencia es
de catorce centavos (p = 0,7543). Estandarizando por mezcla, el recorrido entre las cinco ciudades baja
de 47,71 a 15,24 dólares.

La frase que va al informe no es «Cuenca vende más por factura»: es **«Cuenca tiene más clientes
mayoristas»**. Dos diagnósticos distintos que llevan a dos acciones distintas.

## 4. Chi cuadrado: dos variables categóricas

Cuando las dos variables son categorías —grupo y conversión, canal y devolución— no hay promedios que
comparar. Chi cuadrado compara la tabla observada contra la que saldría si las dos variables fueran
independientes, y mide cuánto se aleja.

In [ ]:
tabla_ab = pd.crosstab(experimento["grupo"], experimento["convirtio"])
tabla_ab.columns = ["no convirtió", "convirtió"]
chi2, p_chi, gl, esp = stats.chi2_contingency(tabla_ab)
v_cramer = np.sqrt(chi2 / tabla_ab.values.sum())

print("Observado:")
print(tabla_ab.to_string())
print("\nEsperado si el grupo no influyera en la conversión:")
print(pd.DataFrame(esp.round(2), index=tabla_ab.index, columns=tabla_ab.columns).to_string())
print(f"\nchi cuadrado = {chi2:,.4f}   gl = {gl}   p = {p_chi:.3g}")
print(f"V de Cramér  = {v_cramer:.4f}   ← tamaño del efecto (0 = ninguna relación, 1 = total)\n")

print("La misma prueba dentro de cada segmento, que es donde la comparación es limpia:")
for tipo in ["Minorista", "Mayorista"]:
    sub = experimento.query("tipo_cliente == @tipo")
    t_sub = pd.crosstab(sub["grupo"], sub["convirtio"])
    c_sub, p_sub, _, _ = stats.chi2_contingency(t_sub)
    tasa = sub.groupby("grupo")["convirtio"].mean()
    print(f"  {tipo:10s} control {tasa['Control']:.4%} vs tratamiento {tasa['Tratamiento']:.4%}  "
          f"chi2 = {c_sub:>7,.4f}  p = {p_sub:.5f}  "
          f"V = {np.sqrt(c_sub / len(sub)):.4f}")

Tres pruebas, tres p-valores por debajo de 0,05 y **dos conclusiones opuestas**. La global
(p = 4,68e-05) declara significativa una diferencia a favor del control que ya sabemos artificial; las
dos por segmento (p = 0,0039 y p = 0,0168) declaran significativa la del tratamiento, que es la real.

⚠️ **El p-valor no protege de la variable de confusión.** Mide si la diferencia observada es compatible
con el azar, no si la comparación tenía sentido. Una prueba bien ejecutada sobre grupos mal formados da
un número impecable y una conclusión falsa.

## 5. ANOVA: tres o más grupos

Comparar cinco ciudades de dos en dos son diez pruebas t, y con diez pruebas al 5 % la probabilidad de
al menos un falso positivo sube al 40 %. ANOVA hace una sola pregunta —¿alguna de las cinco medias se
sale del grupo?— y devuelve un solo p-valor.

In [ ]:
grupos_ciudad = [facturas.query("ciudad == @c")["monto"].values
                 for c in sorted(facturas["ciudad"].unique())]
F, p_anova = stats.f_oneway(*grupos_ciudad)

# eta cuadrado: qué porcentaje de la varianza del ticket explica la ciudad.
todo = np.concatenate(grupos_ciudad)
media_global = todo.mean()
ss_entre = sum(len(g) * (g.mean() - media_global) ** 2 for g in grupos_ciudad)
ss_total = ((todo - media_global) ** 2).sum()
eta2 = ss_entre / ss_total

print(f"ANOVA del ticket entre las cinco ciudades")
print(f"  F = {F:,.4f}   p = {p_anova:.3g}   ← significativo")
print(f"  eta cuadrado = {eta2:.6f}  →  la ciudad explica el {eta2:.4%} de la varianza del ticket\n")

# El supuesto de varianzas iguales, comprobado en lugar de supuesto.
lev, p_lev = stats.levene(*grupos_ciudad)
print(f"prueba de Levene (¿varianzas iguales?): estadístico {lev:,.4f}  p = {p_lev:.3g}")
print("  → las varianzas NO son iguales, así que el ANOVA clásico va con reserva declarada.\n")

print("La misma pregunta dentro de cada tipo de cliente:")
for tipo in ["Minorista", "Mayorista"]:
    sub = facturas.query("tipo_cliente == @tipo")
    gs = [sub.query("ciudad == @c")["monto"].values for c in sorted(sub["ciudad"].unique())]
    F_s, p_s = stats.f_oneway(*gs)
    t_s = np.concatenate(gs); m_s = t_s.mean()
    e2 = sum(len(g) * (g.mean() - m_s) ** 2 for g in gs) / ((t_s - m_s) ** 2).sum()
    print(f"  {tipo:10s} F = {F_s:>8,.4f}   p = {p_s:.4f}   eta cuadrado = {e2:.6f}")

📌 **La ciudad explica el 0,26 % de la varianza del ticket.** El p-valor de 1,90e-08 es de los
que hacen que alguien escriba «diferencia altamente significativa entre ciudades» en la primera línea
del informe. El eta cuadrado dice que el 99,74 % de por qué una factura es grande o pequeña **no tiene
nada que ver con la ciudad**.

Y otra vez, al abrir por tipo de cliente el hallazgo se evapora: entre minoristas el p-valor es 0,9332,
que es tan cerca de «no pasa nada» como se puede llegar. Todo lo que el ANOVA global detectó era la
proporción de mayoristas de cada plaza.

**ANOVA dice que hay una diferencia. No dice cuál, ni de qué tamaño, ni por qué.** Las tres preguntas
que importan quedan fuera.

In [ ]:
# ✅ Comprobación 2 · significativo y minúsculo son cosas distintas
assert p < 0.05, "La prueba t entre Cuenca y Manta tiene que salir significativa"
assert abs(d) < 0.2, \
    f"La d de Cohen debería ser pequeña (< 0,2) y te dio {d:.4f}: revisa la función d_de_cohen"
assert abs(v_cramer) < 0.2, \
    "La V de Cramér del experimento también es pequeña: el efecto global es débil y contaminado"

print(f"Comprobación 2 superada ✓  p = {p:.3g} (significativo) con d = {d:.4f} (minúsculo): "
      "las dos cosas a la vez")

## 6. Diseñar la prueba A/B antes de lanzarla

El experimento de la sección 1 falló por lo que no se escribió antes de empezar. La plantilla de abajo
son los seis campos que se llenan **en ese orden** y antes del primer envío. El más incómodo es el
tercero, porque a veces la respuesta es que la prueba no se puede hacer.

In [ ]:
from scipy.stats import norm


def n_por_grupo(p0, mejora_relativa, alpha=0.05, potencia=0.80):
    """Clientes necesarios EN CADA GRUPO para detectar una mejora relativa sobre p0."""
    p1 = p0 * (1 + mejora_relativa)
    za, zb = norm.ppf(1 - alpha / 2), norm.ppf(potencia)
    p_medio = (p0 + p1) / 2
    numerador = (za * np.sqrt(2 * p_medio * (1 - p_medio))
                 + zb * np.sqrt(p0 * (1 - p0) + p1 * (1 - p1))) ** 2
    return int(np.ceil(numerador / (p1 - p0) ** 2))


def mde_detectable(p0, n, alpha=0.05, potencia=0.80):
    """La mejora relativa más pequeña que n clientes por grupo permiten detectar."""
    bajo, alto = 1e-4, min(5.0, 0.99 / p0 - 1)      # p1 nunca puede pasar de 1
    for _ in range(120):
        medio = (bajo + alto) / 2
        if n_por_grupo(p0, medio, alpha, potencia) > n:
            bajo = medio
        else:
            alto = medio
    return (bajo + alto) / 2


disponibles = experimento["tipo_cliente"].value_counts()
escenarios = []
for tipo, p0 in [("Minorista", tasas.loc["Minorista", "Control"]),
                 ("Mayorista", tasas.loc["Mayorista", "Control"])]:
    n_disp = int(disponibles[tipo] // 2)
    for mejora in [0.20, 0.30, 0.50]:
        escenarios.append((tipo, p0, mejora, n_por_grupo(p0, mejora), n_disp,
                           n_por_grupo(p0, mejora) <= n_disp))

esc = pd.DataFrame(escenarios, columns=["segmento", "conversión base", "mejora buscada",
                                        "n necesario por grupo", "n disponible por grupo",
                                        "¿alcanza?"])
esc["conversión base"] = esc["conversión base"].map(lambda v: f"{v:.2%}")
esc["mejora buscada"] = esc["mejora buscada"].map(lambda v: f"+{v:.0%}")
print(esc.to_string(index=False), "\n")
for tipo, p0 in [("Minorista", tasas.loc["Minorista", "Control"]),
                 ("Mayorista", tasas.loc["Mayorista", "Control"])]:
    n_disp = int(disponibles[tipo] // 2)
    m = mde_detectable(p0, n_disp)
    print(f"{tipo:10s} base {p0:.2%} · {n_disp:,} por grupo → "
          f"mejora mínima detectable {m:.2%} (llegar al {p0 * (1 + m):.2%})")

⚠️ **Con toda la cartera de Comercial Andina no se puede detectar una mejora del 20 % en
minoristas.** Harían falta 7 204 clientes por grupo y solo hay 706. La respuesta honesta no es lanzar
igual y ver qué sale: es **cambiar la pregunta**. Con 706 por grupo solo se detectan mejoras del 70,21 %
o más, y con 194 mayoristas por grupo, del 35,94 % en adelante. Las dos cifras que el experimento real
encontró —+98,8 % relativo en minoristas y +40,1 % en mayoristas— están por encima de esos umbrales, y
por eso salieron significativas pese al tamaño de la cartera.

Ahora la duración y el criterio de parada, que es donde se pierden los experimentos que sí estaban bien
diseñados.

### Plantilla de diseño de una prueba A/B

Los seis campos, llenos para la campaña de reactivación de Comercial Andina. Esta es la tabla que se
entrega **antes** de enviar el primer correo, firmada por quien va a decidir con el resultado.

| campo | qué se escribe | campaña de reactivación |
|---|---|---|
| **1 · Hipótesis** | Una frase falsable, con dirección y magnitud | Enviar el correo de reactivación aumenta la conversión a compra en 30 días de los clientes inactivos |
| **2 · Unidad de asignación** | Qué se sortea: cliente, factura, sucursal, sesión | El **cliente**. Nunca la factura: el mismo cliente caería en los dos grupos y contaminaría la comparación |
| **3 · Aleatorización** | Cómo se sortea y qué se estratifica | Sorteo dentro de cada `tipo_cliente` por separado. Es la corrección directa del fallo que produjo la paradoja |
| **4 · Métrica primaria** | Una sola, definida sin ambigüedad | Proporción de clientes con al menos una compra en los 30 días siguientes al envío |
| **5 · Tamaño de muestra** | n por grupo, con la mejora que se busca detectar | 194 mayoristas por grupo (detecta +35,94 %) y 706 minoristas por grupo (detecta +70,21 %) |
| **6 · Duración y parada** | Fecha de corte y qué se hace con cada resultado | 60 días. Una sola lectura al final. Parada por daño si el tratamiento cae 3 pp bajo el control (el cálculo del reclutamiento y las cinco reglas de parada, en el apéndice) |

Dos campos más que no son obligatorios y salvan proyectos: **métricas de guardia** (¿sube la tasa de
devolución? ¿se dispara la baja de suscripción?) y **qué NO se va a mirar**, para no salir a buscar el
segmento en el que sí funcionó.

### 🌶️ Ejercicio 1 — Guiado

Busca una segunda paradoja de Simpson en los mismos datos, ahora con la **ciudad** como tercer factor.
Calcula la conversión por grupo dentro de cada una de las cinco ciudades, comprueba si en alguna se
invierte el signo respecto del global, y explica en dos líneas si esa inversión es real o es la mezcla
de mayoristas de esa plaza. Cierra con una prueba de chi cuadrado por ciudad.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: experimento.groupby(["ciudad", "grupo"])["convirtio"].mean().unstack()
# Pista 2: para saber si es mezcla, mira pd.crosstab(experimento["ciudad"], experimento["tipo_cliente"],
#          normalize="index") y compárala con la mezcla global
# Pista 3: con cinco ciudades son cinco pruebas; si haces cinco al 5 %, la probabilidad de al menos
#          un falso positivo es 1 - 0.95**5. Escribe ese número antes de interpretar los p-valores

### 🔥 Desafío

El gerente comercial afirma: **«los clientes captados por visita comercial compran más que los captados
por redes sociales»**. Contrástalo con datos. Necesitas: (a) la comparación cruda con una prueba t y su
tamaño del efecto, (b) al menos dos variables de confusión candidatas —piensa en qué tipo de cliente
capta cada canal y desde cuándo—, (c) la misma comparación controlando por la confusión que encuentres,
y (d) una frase que le puedas decir al gerente sin mentir en ninguna dirección.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: el canal de captación está en clientes.csv; únelo a la facturación por cliente
# Pista 2: la confusión más obvia es tipo_cliente. La segunda, la antigüedad: un cliente captado
#          hace tres años tuvo más tiempo de comprar que uno de hace tres meses
# Pista 3: para controlar por antigüedad, compara facturación POR AÑO de relación, no acumulada
# Pista 4: reporta siempre p-valor y d de Cohen juntos. Si la d es menor que 0,2, dilo

### 🎯 Reto en clase (15 min)

En equipos y contra reloj: **inventen la variable de confusión**. Cada equipo recibe una afirmación
causal de negocio —«el programa de fidelidad hace que los clientes gasten el doble», «las tiendas más
grandes venden más por metro cuadrado», «los clientes que abren el correo compran más»— y tiene quince
minutos para escribir tres explicaciones alternativas que produzcan el mismo dato sin que la causa sea
la declarada, y decir con qué dato se descartaría cada una. Gana el equipo que proponga la explicación
alternativa que el equipo dueño de la afirmación no pueda descartar con los datos que tiene.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: la causalidad inversa es la explicación alternativa que más se olvida.
#   «los del programa de fidelidad gastan el doble» → ¿el programa los hizo gastar,
#    o los que ya gastaban fueron los que se inscribieron?
# Molde para atacar cualquiera de las tres afirmaciones:
#   (facturas
#    .groupby(["<grupo que se compara>", "<confusión candidata>"])["monto"]
#    .agg(["size", "mean"])
#    .unstack())

## La trampa de hoy

⚠️ **Declarar un hallazgo porque el p-valor bajó de 0,05, sin mirar cuánto cambia la cifra de negocio.**
Con suficientes datos, absolutamente todo es significativo. Y «suficientes datos» hoy es cualquier
extracción de un ERP.

La demostración es directa: tomamos una diferencia que en la muestra real **no es significativa** —el
ticket minorista en línea contra el de tienda— y la repetimos idéntica, sin cambiar ni un centavo, con
muestras cada vez más grandes.

In [ ]:
minoristas = facturas.query("tipo_cliente == 'Minorista'")
en_linea = minoristas.query("canal == 'Online'")["monto"].values
en_tienda = minoristas.query("canal == 'Tienda'")["monto"].values

filas = []
for k in [1, 2, 5, 10, 20, 50]:
    a, b = np.tile(en_linea, k), np.tile(en_tienda, k)
    t_k, p_k = stats.ttest_ind(a, b, equal_var=False)
    filas.append((k, len(a) + len(b), a.mean() - b.mean(),
                  d_de_cohen(pd.Series(a), pd.Series(b)), p_k, p_k < 0.05))

trampa = pd.DataFrame(filas, columns=["muestra ×", "n total", "diferencia (dólares)",
                                      "d de Cohen", "p-valor", "¿significativa?"])
print("La MISMA diferencia de 12 centavos, con muestras cada vez más grandes:\n")
print(trampa.to_string(index=False, float_format=lambda x: f"{x:,.6g}"))
print(f"\ndiferencia real          : {en_linea.mean() - en_tienda.mean():,.4f} dólares por factura")
print(f"tamaño del efecto        : {d_de_cohen(pd.Series(en_linea), pd.Series(en_tienda)):.4f} "
      f"(la d NO cambia con el tamaño de muestra)")
print(f"p-valor con la muestra real (n = {len(en_linea) + len(en_tienda):,}) : "
      f"{stats.ttest_ind(en_linea, en_tienda, equal_var=False)[1]:.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trampa["n total"], trampa["p-valor"], marker="o", color="#C44E52", label="p-valor")
ax.axhline(0.05, color="#4C72B0", linestyle="--", label="umbral 0,05")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("tamaño de muestra (escala logarítmica)")
ax.set_ylabel("p-valor (escala logarítmica)")
ax.set_title("La diferencia sigue siendo de 12 centavos: lo único que cambió es el número de filas")
for _, r in trampa.iterrows():
    ax.annotate(f"d={r['d de Cohen']:.4f}", (r["n total"], r["p-valor"]),
                textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8)
ax.legend()
plt.tight_layout()
plt.show()

📌 **La diferencia es de 0,1284 dólares. Doce centavos.** Con la muestra real (10 385 facturas) el
p-valor es 0,5956 y nadie reportaría nada. Repitiendo esos mismos datos veinte veces cae a 0,0176 y la
diferencia pasa a ser «estadísticamente significativa»; con cincuenta llega a 0,000174 y se escribe
«altamente significativa». **No se agregó ni un dato nuevo.** La d de Cohen, que sí mide lo que importa,
se queda clavada en 0,0110 en las seis filas.

El p-valor mezcla dos cosas —el tamaño del efecto y el tamaño de la muestra— y con muestras grandes
solo informa de la segunda. La defensa es una regla de redacción: **ninguna frase de un informe lleva
un p-valor si no lleva al lado la magnitud en unidades de negocio.**

La versión operativa, para pegar en la pared del equipo: **¿es significativo?** (¿se descarta el azar?),
**¿es grande?** (¿cuánto cambia la cifra que le importa al gerente?) y **¿es comparable?** (¿los dos
grupos se parecen en todo lo demás?). Las tres, siempre, en ese orden. La tercera es la que produjo la
paradoja de la sección 2.

## Entregable

Sube `lab_09_apellido.ipynb` con:

- La paradoja de Simpson documentada con las cinco cifras: 21,76 % contra 14,15 % en el global, 5,62 %
  contra 11,17 % en minoristas, 39,35 % contra 55,13 % en mayoristas, el 79,90 % de mayoristas
  asignados al control y el efecto estandarizado de +7,76 puntos.
- La conclusión de negocio en tres frases, incluida la que dice cómo se aleatoriza el próximo
  experimento.
- Las pruebas —t, chi cuadrado global, chi cuadrado por segmento y ANOVA— cada una con su p-valor, su
  tamaño del efecto y **una frase de traducción al negocio**. Sin la frase, la prueba no cuenta.
- La demostración de la trampa: la misma diferencia de 0,1284 dólares pasando de p = 0,5956 a
  p = 0,000174 solo por multiplicar la muestra.
- La plantilla de diseño A/B del caso con los seis campos llenos, con el tamaño de muestra calculado
  con `n_por_grupo` y la frase de qué se hace con cada resultado posible.
- Una fila nueva en la bitácora de prompts: le pediste al asistente que enumerara todas las variables
  de confusión posibles de tu hipótesis. Anota cuáles descartaste y por qué; las que sobrevivan van a
  las limitaciones del proyecto.

## Para tu equipo

- La prueba A/B del caso se diseña esta semana con presupuesto y duración **reales**. Si al calcular el
  tamaño de muestra descubren que la empresa no tiene suficientes clientes —que es lo más probable— la
  entrega correcta no es una prueba imposible: es una pregunta más pequeña que sí se puede contestar,
  o una mejora buscada más grande, escrita y justificada.
- Revisen los hallazgos de las semanas 5 a 8 buscando comparaciones sin control. Toda frase del tipo
  «el segmento X compra más que el Y» necesita que alguien pregunte «¿comparados en igualdad de qué?».
  Al menos una de sus conclusiones anteriores no va a sobrevivir, y encontrarla ahora vale más que
  defenderla en la semana 16.
- El diseño experimental de una página es la entrega de la semana. Se escribe **antes** de tener el
  resultado; un diseño redactado después de ver los datos no es un diseño, es una explicación.

## Apéndice · para profundizar fuera de clase

Lo que sigue **no compite por los noventa minutos de clase**: es opcional y está aquí como material de
consulta para el diseño experimental del proyecto. Se ejecuta después de haber corrido todas las celdas
anteriores.

### A0 · La paradoja de Simpson en dos gráficos

Las mismas cifras de la sección 2, dibujadas: a la izquierda el resultado global y a la derecha el mismo experimento abierto por tipo de cliente. Es la figura que se lleva a la reunión cuando hay que explicar por qué la recomendación cambia.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))

x = np.arange(2)
axes[0].bar(x - 0.2, [global_.loc["Control", "tasa"] * 100, estandarizado["Control"] * 100],
            0.4, label="Control", color="#B0B0B0")
axes[0].bar(x + 0.2, [global_.loc["Tratamiento", "tasa"] * 100, estandarizado["Tratamiento"] * 100],
            0.4, label="Tratamiento", color="#4C72B0")
axes[0].set_xticks(x, ["lectura ingenua", "estandarizada por mezcla"])
axes[0].set_ylabel("conversión %")
axes[0].set_title("Con los mismos datos, la campaña pierde o gana\nsegún se controle la mezcla",
                  fontsize=11)
axes[0].legend()

seg = ["Minorista", "Mayorista"]
xs = np.arange(len(seg))
axes[1].bar(xs - 0.2, tasas.loc[seg, "Control"] * 100, 0.4, label="Control", color="#B0B0B0")
axes[1].bar(xs + 0.2, tasas.loc[seg, "Tratamiento"] * 100, 0.4, label="Tratamiento", color="#4C72B0")
for i, s in enumerate(seg):
    axes[1].text(i + 0.2, tasas.loc[s, "Tratamiento"] * 100 + 1,
                 f"{tasas.loc[s, 'diferencia pp']:+.1f} pp", ha="center", fontsize=10)
axes[1].set_xticks(xs, seg)
axes[1].set_ylabel("conversión %")
axes[1].set_title("Abierto por segmento, el tratamiento gana en los dos", fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

### A1 · Las cuatro pruebas, lado a lado

Elegir la prueba es mecánico y cabe en cuatro líneas. Lo que no es mecánico —y es lo que se
califica— es la columna de la derecha.

In [ ]:
guia = pd.DataFrame([
    ("prueba t", "una numérica, dos grupos", "¿el ticket difiere entre Cuenca y Manta?",
     "d de Cohen"),
    ("chi cuadrado", "dos categóricas", "¿la conversión depende del grupo?", "V de Cramér"),
    ("ANOVA", "una numérica, tres o más grupos", "¿el ticket difiere entre las cinco ciudades?",
     "eta cuadrado"),
    ("proporciones", "una categórica, dos grupos", "¿la tasa de devolución difiere por canal?",
     "diferencia en puntos"),
], columns=["prueba", "cuándo se usa", "ejemplo en Comercial Andina", "tamaño del efecto"])
print(guia.to_string(index=False), "\n")

resultados = pd.DataFrame([
    ("t · Cuenca vs Manta", p, "d de Cohen", d,
     f"{cuenca.mean() - manta.mean():,.2f} dólares de ticket"),
    ("chi2 · conversión global", p_chi, "V de Cramér", v_cramer,
     f"{dif * 100:+.2f} pp (contaminado por la mezcla)"),
    ("chi2 · conversión minoristas",
     stats.chi2_contingency(pd.crosstab(
         experimento.query("tipo_cliente == 'Minorista'")["grupo"],
         experimento.query("tipo_cliente == 'Minorista'")["convirtio"]))[1],
     "diferencia", tasas.loc["Minorista", "diferencia pp"] / 100,
     f"{tasas.loc['Minorista', 'diferencia pp']:+.2f} pp de conversión"),
    ("ANOVA · cinco ciudades", p_anova, "eta cuadrado", eta2,
     f"{tabla['ticket observado'].max() - tabla['ticket observado'].min():,.2f} de recorrido"),
], columns=["prueba", "p-valor", "métrica de efecto", "valor", "traducción al negocio"])
resultados["significativa"] = resultados["p-valor"] < 0.05
print(resultados.to_string(index=False, float_format=lambda x: f"{x:,.6g}"))
print("\nLas cuatro son significativas. Solo dos cambian una decisión.")

**Regla de lectura, sin excepciones: el p-valor decide si el hallazgo se reporta; el tamaño del
efecto decide si se hace algo.** Los dos van siempre juntos en la misma frase.

Referencia rápida de la d de Cohen: 0,2 pequeño · 0,5 medio · 0,8 grande. Y una advertencia que vale
más que la tabla: esos umbrales son convenciones, no leyes. En un negocio con margen del 44 %, un
efecto «pequeño» sobre un millón de facturas puede valer más que uno «grande» sobre doscientas.

### A2 · Reclutamiento, duración y criterio de parada

El tamaño de muestra dice cuántos hacen falta; esto dice **cuánto se tarda en juntarlos** y cuándo se para. Las cinco reglas de parada se escriben antes del primer envío y se firman: son las que impiden que alguien mire el resultado cada mañana hasta que salga favorable.

In [ ]:
ELEGIBLES_MES = int(len(experimento) / 6)     # supuesto declarado: la cartera entra en seis oleadas
CONVERSIONES_DIA = 12                        # supuesto declarado: capacidad de atención comercial

for tipo, p0 in [("Minorista", tasas.loc["Minorista", "Control"]),
                 ("Mayorista", tasas.loc["Mayorista", "Control"])]:
    n_disp = int(disponibles[tipo] // 2)
    total = n_disp * 2
    semanas = np.ceil(total / (ELEGIBLES_MES / 4))
    print(f"{tipo:10s} {total:,} clientes en el experimento · "
          f"{semanas:.0f} semanas para reclutarlos a {ELEGIBLES_MES / 4:.0f} por semana")

print(f"\nDuración mínima = tiempo de reclutamiento + ventana de conversión (30 días).")
print("Criterio de parada, escrito ANTES de mirar el resultado:")
reglas = pd.DataFrame([
    ("Fecha de corte", "se para el día 60, haya pasado lo que haya pasado"),
    ("Nada de mirar a diario", "una sola lectura al final; mirar cada día y parar al primer p < 0,05 "
     "multiplica por tres los falsos positivos"),
    ("Parada por daño", "si la conversión del tratamiento cae más de 3 pp bajo el control en la "
     "primera semana, se detiene y se revisa el envío"),
    ("Métrica primaria", "conversión a compra en 30 días. Una sola. Las demás son secundarias "
     "y no deciden nada"),
    ("Qué se hace con cada resultado", "gana → despliegue a toda la cartera · empata → no se "
     "despliega y se documenta el costo · pierde → se apaga"),
], columns=["regla", "contenido"])
print(reglas.to_string(index=False))